# Lipid tail splay - DOPC/DOPA composition series

Intra-lipid splay = angle between a lipid's two acyl-chain vectors
(sn-1: first->last C3x carbon; sn-2: first->last C2x carbon). We build the
probability distribution P(theta) per composition, per lipid type, with an
error band from block averaging - same style as the RDF figure.

**Trajectory to use:** the full-atom **whole (+centered)** xtc, e.g.
`Prod_merged_center_*.xtc`. Splay is an internal angle, so molecules only need to
be *whole*; centering is harmless. Do **not** use the nojump trajectory (that is
only for diffusion), and do not use the P-only `heads` trajectory (splay needs
the chain carbons).

## Setup

In [ ]:
# !pip install "MDAnalysis>=2.4" numpy matplotlib
%matplotlib inline
import os, re
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

## Config - edit the paths

In [ ]:
LIPIDS   = ["DOPC", "DOPA"]    # the two lipids -> one panel each
BINW_DEG = 2.0                 # splay-angle bin width (degrees)
ANG_MAX  = 180.0               # angle axis range
NBLOCKS  = 5                   # time blocks for the error band
STEP     = 1                   # frame stride

# label -> (topology, full-atom trajectory[whole+centered], DOPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#2ca02c",   # green
    "72/28": "#1f77b4",   # blue
    "80/20": "#ff7f0e",   # orange
    "87/13": "#9467bd",   # purple
    "95/5" : "#d62728",   # red
}
os.makedirs("results", exist_ok=True)

## Splay core (block-averaged distribution)

In [ ]:
# Splay core (block-averaged distribution)
def _carbons(res, prefix):
    # acyl carbons of a chain ('C3'=sn-1, 'C2'=sn-2), skipping carbonyl idx 1
    pat = re.compile(r"^" + prefix + r"(\d+)$")
    hits = [(int(m.group(1)), at) for at in res.atoms
            if (m := pat.match(at.name)) and int(m.group(1)) >= 2]
    return sorted(hits, key=lambda t: t[0])

def _splay_indices(u, resname):
    # (N,4) atom indices: sn1_first, sn1_last, sn2_first, sn2_last per lipid
    idx = []
    for res in u.select_atoms(f"resname {resname}").residues:
        c3 = _carbons(res, "C3")      # sn-1
        c2 = _carbons(res, "C2")      # sn-2
        if len(c3) >= 2 and len(c2) >= 2:
            idx.append((c3[0][1].index, c3[-1][1].index,
                        c2[0][1].index, c2[-1][1].index))
    return np.array(idx, dtype=int)

def splay_blocks(u, resname, binw=BINW_DEG, angmax=ANG_MAX, nblocks=NBLOCKS, step=STEP):
    idx = _splay_indices(u, resname)
    edges = np.arange(0.0, angmax + binw, binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    if len(idx) == 0:
        return ctr, np.zeros_like(ctr), np.zeros_like(ctr), np.nan
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    Hs, all_ang = [], []
    for blk in blocks:
        ang = []
        for fi in blk:
            u.trajectory[int(fi)]
            pos = u.atoms.positions
            v1 = pos[idx[:, 1]] - pos[idx[:, 0]]      # sn-1 vector
            v2 = pos[idx[:, 3]] - pos[idx[:, 2]]      # sn-2 vector
            n1 = np.linalg.norm(v1, axis=1); n2 = np.linalg.norm(v2, axis=1)
            good = (n1 > 0) & (n2 > 0)
            cos = np.einsum("ij,ij->i", v1, v2) / (n1 * n2)
            ang.append(np.degrees(np.arccos(np.clip(cos[good], -1, 1))))
        ang = np.concatenate(ang); all_ang.append(ang)
        Hs.append(np.histogram(ang, bins=edges, density=True)[0])
    H = np.array(Hs)
    mean = H.mean(axis=0)
    std = H.std(axis=0) if len(H) > 1 else np.zeros_like(mean)
    return ctr, mean, std, np.concatenate(all_ang).mean()

## Compute for every composition and lipid

In [ ]:
results = {lip: {} for lip in LIPIDS}
means = {}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})"); continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for lip in LIPIDS:
        theta, P, Pstd, mang = splay_blocks(u, lip)
        results[lip][label] = (theta, P, Pstd); means[(lip, label)] = mang
        np.savetxt(f"results/splay_{lip}_{label.replace('/', '-')}.dat",
                   np.column_stack([theta, P, Pstd]), header="angle_deg  P  std")

print("\nMean splay angle (deg):")
print(f"{'comp':<8}" + "".join(f"{lip:>10}" for lip in LIPIDS))
for label in SYSTEMS:
    print(f"{label:<8}" + "".join(f"{means.get((lip,label), float('nan')):10.1f}" for lip in LIPIDS))

## Plot - one panel per lipid, one distinct colour per composition

In [ ]:
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, len(LIPIDS), figsize=(4.6 * len(LIPIDS), 3.8), sharex=True)
if len(LIPIDS) == 1: axes = [axes]
for ax, lip in zip(axes, LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]: continue
        theta, P, Pstd = results[lip][L]
        ax.fill_between(theta, P - Pstd, P + Pstd, color=colors[L], alpha=0.18, lw=0)
        ax.plot(theta, P, color=colors[L], lw=1.8, label=f"{L}  ({SYSTEMS[L][2]}% DOPA)")
    ax.set_xlim(0, ANG_MAX); ax.set_xlabel("Tail splay angle (deg)")
    ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="right", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)
axes[0].set_ylabel("P(θ)")
handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.10))
fig.suptitle("DOPC/DOPA intra-lipid tail splay vs composition", y=1.16)
plt.tight_layout()
plt.savefig("results/splay_DOPC_DOPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

## GROMACS route (equivalent) - `gmx gangle`

Splay = angle between the sn-1 and sn-2 chain vectors. `gmx gangle -g1 vector
-g2 vector` pairs vector *i* of group1 with vector *i* of group2; each vector is
a pair of atoms (chain first & last carbon), so the selections below give one
vector per lipid. DOPC/DOPA are 18:1 -> terminal carbons C318 (sn-1), C218 (sn-2).
(For DLPG/DLPA use C312 / C212.)

```bash
# one composition (repeat for 65,72,80,87,95), one lipid at a time:
TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc

# DOPC splay
gmx gangle -s $TPR -f $XTC -g1 vector -g2 vector \
    -group1 'name C32 C318 and resname DOPC' \
    -group2 'name C22 C218 and resname DOPC' \
    -oh splay_DOPC_65.xvg -binw 2

# DOPA splay
gmx gangle -s $TPR -f $XTC -g1 vector -g2 vector \
    -group1 'name C32 C318 and resname DOPA' \
    -group2 'name C22 C218 and resname DOPA' \
    -oh splay_DOPA_65.xvg -binw 2
```

`-oh` writes the normalized histogram P(theta). For the error band, run on time
blocks (`-b/-e`) and take mean/std across the .xvg files, or use the Python block
averaging above.

## (Optional) plot straight from gmx gangle .xvg files

In [ ]:
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()] for l in open(path) if not l.startswith(("#","@"))]
    d = np.array(rows); return d[:, 0], d[:, 1]     # angle(deg), P

# gmx_files = {"DOPC": {"65/35": "splay_DOPC_65.xvg"}, "DOPA": {"65/35": "splay_DOPA_65.xvg"}}
# fig, axes = plt.subplots(1, len(LIPIDS), figsize=(4.6*len(LIPIDS), 3.8))
# for ax, lip in zip(np.atleast_1d(axes), LIPIDS):
#     for L in labels_sorted:
#         p = gmx_files.get(lip, {}).get(L)
#         if p and os.path.exists(p):
#             x, y = read_xvg_xy(p); ax.plot(x, y, color=colors[L], lw=1.8, label=L)
#     ax.set_xlabel("Tail splay angle (deg)"); ax.set_xlim(0, ANG_MAX)
#     ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic", ha="right", va="top")
# plt.legend(); plt.tight_layout(); plt.show()